# split PARCHG


In [1]:
from pathlib import Path
import math

def _tokens_are_ints(tokens):
    try: return [int(x) for x in tokens]
    except ValueError:
        return None

def _find_counts_line(lines):
    # Find the atom-count line in a POSCAR/PARCHG header.
    # Supports both VASP 4 and VASP 5 style headers.
    possible_counts = _tokens_are_ints(lines[6].split())
    if possible_counts is not None:
        return 6, possible_counts
    possible_counts = _tokens_are_ints(lines[5].split())
    if possible_counts is not None:
        return 5, possible_counts
    raise RuntimeError("Could not identify the atom-count line.")

def _find_first_grid_line(lines):
    # Find the first volumetric grid line after atomic coordinates.
    counts_idx, counts = _find_counts_line(lines)
    n_atoms = sum(counts)
    coord_type_idx = counts_idx + 1
    # Optional Selective dynamics line
    if lines[coord_type_idx].strip().lower().startswith("s"):
        coord_type_idx += 1
    coord_start_idx = coord_type_idx + 1
    after_coords_idx = coord_start_idx + n_atoms
    for i in range(after_coords_idx, len(lines)):
        tokens = lines[i].split()
        grid = _tokens_are_ints(tokens)
        if grid is not None and len(grid) == 3 and all(x > 1 for x in grid):
            return i, grid
    raise RuntimeError("Could not find the first volumetric grid line.")

def _read_n_values(lines, start_idx, n_values):
    # Read n_values floating-point numbers starting after a grid line.
    values = []
    i = start_idx
    while i < len(lines) and len(values) < n_values:
        values.extend(lines[i].split())
        i += 1
    if len(values) < n_values:
        raise RuntimeError("Not enough volumetric data values found.")
    return [float(x) for x in values[:n_values]], i

def _find_next_same_grid(lines, start_idx, grid):
    # Find the next occurrence of the same grid line.
    # In spin-polarized PARCHG, this usually starts the magnetization block.
    for i in range(start_idx, len(lines)):
        tokens = lines[i].split()
        candidate = _tokens_are_ints(tokens)
        if candidate == grid:
            return i
    raise RuntimeError(
        "Could not find the second volumetric block. "
        "This PARCHG may not contain spin information."
    )

def _write_volumetric_file(output_path, header_lines, grid_line, data):
    # Write a CHGCAR/PARCHG-like volumetric file that VESTA can open directly.
    with open(output_path, "w") as f:
        for line in header_lines:
            f.write(line)
        f.write(grid_line)
        for i in range(0, len(data), 5):
            f.write(" ".join(f"{x: .11E}" for x in data[i:i + 5]) + "\n")

def split_parchg_spin(input_parchg="PARCHG", output_up="PARCHG_UP", output_dn="PARCHG_DN"):
    # Split a spin-polarized PARCHG into spin-up and spin-down volumetric files.
    # The input PARCHG is assumed to contain:
    #  first block  = rho_up + rho_down
    #  second block = rho_up - rho_down
    input_parchg = Path(input_parchg).expanduser().resolve()
    output_up = Path(output_up).expanduser().resolve()
    output_dn = Path(output_dn).expanduser().resolve()
    if not input_parchg.exists():
        raise FileNotFoundError(f"File not found: {input_parchg}")
    lines = input_parchg.read_text().splitlines(keepends=True)
    first_grid_idx, grid = _find_first_grid_line(lines)
    n_grid = math.prod(grid)
    # First block: rho_up + rho_down
    rho_tot, after_first_block_idx = _read_n_values(lines, first_grid_idx + 1, n_grid)
    # Second block: rho_up - rho_down
    second_grid_idx = _find_next_same_grid(lines, after_first_block_idx, grid)
    mag, _ = _read_n_values(lines, second_grid_idx + 1, n_grid)
    rho_up = [(rt + m) / 2.0 for rt, m in zip(rho_tot, mag)]
    rho_dn = [(rt - m) / 2.0 for rt, m in zip(rho_tot, mag)]
    header_lines = lines[:first_grid_idx]
    grid_line = lines[first_grid_idx]
    _write_volumetric_file(output_up, header_lines, grid_line, rho_up)
    _write_volumetric_file(output_dn, header_lines, grid_line, rho_dn)
    print("Done.")
    print(f"Input      : {input_parchg}")
    print(f"Output up  : {output_up}")
    print(f"Output down: {output_dn}")
    print(f"Grid       : {grid[0]} x {grid[1]} x {grid[2]}")
    print(f"rho_up min = {min(rho_up): .6E}")
    print(f"rho_up max = {max(rho_up): .6E}")
    print(f"rho_dn min = {min(rho_dn): .6E}")
    print(f"rho_dn max = {max(rho_dn): .6E}")

# Run in the current folder
split_parchg_spin()


Done.
Input      : /Users/lu/Repos/o-B14_20241024/3.1_bandstructure/monolayer_AFM_LPARD_LUS/PARCHG
Output up  : /Users/lu/Repos/o-B14_20241024/3.1_bandstructure/monolayer_AFM_LPARD_LUS/PARCHG_UP
Output down: /Users/lu/Repos/o-B14_20241024/3.1_bandstructure/monolayer_AFM_LPARD_LUS/PARCHG_DN
Grid       : 96 x 100 x 384
rho_up min = -3.651400E-01
rho_up max =  2.240800E+02
rho_dn min = -3.651400E-01
rho_dn max =  2.240800E+02
